# Generador de entidades del chatbot — Admisión y Nivelación UG

**Alcance del notebook (RF-05):** este notebook genera `entidades.json`, que contiene ÚNICAMENTE lo que el motor de extracción por expresiones regulares necesita para reconocer entidades dentro de un mensaje del usuario: nombres/alias de facultades, carreras, procesos y documentos, más los patrones de cédula y fecha.


**Nota**
El contenido informativo más amplio (materias por bloque, fichas completas de carreras, cronograma, cuotas de admisión, información institucional) **No forma parte**, se genera en `03_generar_base_conocimiento.ipynb`, debido que ese contenido no se extrae de la consulta del usuario sino que se usa para redactar las respuestas predefinidas del corpus (RF-01), no para RF-05.

#1) Celda de Importaciones

In [2]:
import json
import os
import unicodedata

from google.colab import files

#2) Nombre del archivo

In [3]:
ARCHIVO_SALIDA = "entidades.json"

#3) Plantilla de Funciones

Molde para cada tipo de entidad: solo `id`, `nombre` y `alias` (las variantes textuales que el usuario podría escribir).

Nada de descripciones largas ni datos narrativos, eso pertenece a la base de conocimiento, no a una entidad detectable por RegEx.

In [4]:
def crear_entidad_simple(
    id, nombre,
    alias=None):

    """Molde genérico para entidades cuyo único propósito es ser reconocidas
    dentro del texto del usuario (facultad, proceso, documento)."""

    return {
        "id": id,
        "nombre": nombre,
        "alias": alias if alias else [nombre.lower()],
    }


def crear_referencia_carrera(
    id,
    nombre,
    facultad_id,
    bloque_id,
    alias=None
    ):

    """
    Referencia de una carrera para fines de reconocimiento textual:
    id, nombre, alias y a qué facultad/bloque pertenece
    (útil para que el programa busque luego su ficha completa en base_conocimiento.json).

    No incluye descripción, duración, modalidad, etc. Eso yase en la base
    de conocimiento.
    """

    return {
        "id": id,
        "nombre": nombre,
        "facultad_id": facultad_id,
        "bloque_id": bloque_id,
        "alias": alias if alias else [nombre.lower()],
    }


#Diccionarios para la construcción de las celdas de las carreras
REFERENCIAS_B1 = {}
REFERENCIAS_B2 = {}
REFERENCIAS_B3 = {}
REFERENCIAS_B4 = {}
REFERENCIAS_B5 = {}
REFERENCIAS_B6 = {}


# 4) Facultades

In [6]:
facultad_arquitectura_urbanismo = crear_entidad_simple(
    id="FAU",
    nombre="Facultad de... Arquitectura y Urbanismo",
    alias=["fau", "arquitectura y urbanismo", "Facultad de Arquitectura y Urbanismo"]
)

facultad_fcmf = crear_entidad_simple(
    id="FCMF",
    nombre="Facultad de Ciencias Matemáticas y Físicas",
    alias=["fcmf", "matemáticas y físicas", "Facultad de Ciencias Matemáticas y Físicas"]
)

facultad_industrial = crear_entidad_simple(
    id="FII",
    nombre="Facultad de Ingeniería Industrial",
    alias=["fi", "industrial", "Facultad de Ingeniería Industrial", "fii"]
)

facultad_ciencias_agrarias = crear_entidad_simple(
    id="FCCAGRA",
    nombre="Facultad Ciencias Agrarias",
    alias=["agraria", "ciencias agrarias", "ciencia agraria", "ciencias agraria", "ciencia agrarias", "Facultad Ciencias Agrarias"]
)

facultad_ciencias_naturales = crear_entidad_simple(
    id="FCCNN",
    nombre="Facultad Ciencias Naturales",
    alias=["fccnn", "ciencia naturales", "ciencias naturales", "ciencia natural", "Facultad Ciencias Naturales"]
)

facultad_quimica = crear_entidad_simple(
    id="FIQ",
    nombre="Facultad de Ingeniería Química",
    alias=["fiq", "quimica", "ingenieria quimica", "ing quimica", "facultad quimica", "facultad de quimica", "Facultad de Ingeniería Química"]
)

facultad_ciencias_quimicas = crear_entidad_simple(
    id="FCCQQ",
    nombre="Facultad de Ciencias Químicas",
    alias=["fccqq", "Facultad de Ciencias Químicas", "ciencias quimicas", "ciencia quimica", "ciencia quimicas"]
)

facultad_jurisprudencia = crear_entidad_simple(
    id="FJCSP",
    nombre="Facultad de Jurisprudencia, Ciencias Sociales y Políticas",
    alias=["fjcsp", "jurisprudencia", "ciencia sociales", "ciencias sociales", "ciencias politicas", "ciencia politica", "Facultad de Jurisprudencia, Ciencias Sociales y Políticas"]
)

facultad_ciencias_administrativa = crear_entidad_simple(
    id="FCCAA",
    nombre="Facultad de Ciencias Administrativas",
    alias=["fccaa", "ciencias administrativas", "ciencia administrativa", "Facultad de Ciencias Administrativas"]
)

facultad_medicina_veterinaria_zootecnia = crear_entidad_simple(
    id="FMVZ",
    nombre="Facultad de Medicina Veterinaria y Zootecnia",
    alias=[
        "fmvz",
        "Facultad de Medicina Veterinaria y Zootecnia",
        "zootecnia",
        "veterinaria",
        "facultad de veterinaria",
        "facultad de medicina veterinaria",
        "facultad de medicina de zootecnia",
        "facultad de zootecnia"
    ]
)

facultad_ciencia_comunicacion_social = crear_entidad_simple(
    id="FACSO",
    nombre="Facultad de Ciencias de la Comunicación Social",
    alias=[
        "facso",
        "Facultad de Ciencias de la Comunicación Social",
        "ciencia social",
        "facultad ciencias sociales",
        "facultad ciencia social",
        "facultad de comunicacion",
        "comunicación",
        "comunicación social"
    ]
)

facultad_filosofia_letras_educacion = crear_entidad_simple(
    id="FILO",
    nombre="Facultad de Filosofía, Letras y Ciencias de la Educación",
    alias=["filo", "Facultad de Filosofía, Letras y Ciencias de la Educación", "facultad de educacion", "filosofia y letras", "facultad de filosofia y letras", "ciencias de la educación"]
)

facultad_actividad_fisica = crear_entidad_simple(
    id="FACAF",
    nombre="Facultad de Ciencias de la Actividad Física",
    alias=[
        "facaf",
        "Facultad de Ciencias de la Actividad Física",
        "deporte",
        "actividad física",
        "facultad de actividad física"
    ]
)

facultad_ciencias_psicologicas = crear_entidad_simple(
    id="FCP",
    nombre="Facultad de Ciencias Psicológicas",
    alias=["fcp", "Facultad de Ciencias Psicológicas", "facultad d psicología", "facultad de ciencia psicologica", "facultad de ciencias psicologica"]
)

facultad_economia = crear_entidad_simple(
    id="FCE",
    nombre="Facultad de Economía",
    alias=[
        "fce",
        "economia",
        "economista",
        "negocios",
        "Facultad de Economía"
    ]
)

facultad_ciencias_medicas = crear_entidad_simple(
    id="FCCMM",
    nombre="Facultad de Ciencias Médicas",
    alias=["fccmm", "Facultad de Ciencias Médicas", "Facultad de medicina", "ciencias médicas"]
)

facultad_odontologia = crear_entidad_simple(
    id="FO",
    nombre="Facultad Piloto de Odontología",
    alias=["fou", "odontologia", "odontologo", "dientes", "facultad de odontologia", "Facultad Piloto de Odontología"]
)



FACULTADES = [
    facultad_arquitectura_urbanismo,
    facultad_fcmf,
    facultad_industrial,
    facultad_ciencias_agrarias,
    facultad_ciencias_naturales,
    facultad_quimica,
    facultad_ciencias_quimicas,
    facultad_jurisprudencia,
    facultad_ciencias_administrativa,
    facultad_medicina_veterinaria_zootecnia,
    facultad_ciencia_comunicacion_social,
    facultad_filosofia_letras_educacion,
    facultad_actividad_fisica,
    facultad_ciencias_psicologicas,
    facultad_economia,
    facultad_ciencias_medicas,
    facultad_odontologia
]

# 5) Carreras

Representa la lista de las 59 carreras, con su `facultad_id` y `bloque_id` como referencia — pero sin ficha completa (datos que están  `03_generar_base_conocimiento.ipynb`).


El `bloque_id` identifica a qué bloque de conocimiento pertenece cada una según la tabla oficial de asignaturas evaluadas. (Página web: https://admision.ug.edu.ec/admision/) - Apartado de Evaluación

## Bloque 1 - Ciencia, Tecnología e Ingeniería

In [7]:
REFERENCIAS_B1["ARQ"] = crear_referencia_carrera(
    id="ARQ",
    nombre="Arquitectura",
    facultad_id="FAU",
    bloque_id="B1",
    alias=["arquitectura", "arq"]
)

REFERENCIAS_B1["CDDEIA"] = crear_referencia_carrera(
    id="CDDEIA",
    nombre="Ciencias de Datos e Inteligencia Artificial",
    facultad_id="FCMF",
    bloque_id="B1",
    alias=["ciencias de datos", "ciencia de datos e ia", "datos e ia", "datos", "ia"]
)

REFERENCIAS_B1["ING-CIVIL"] = crear_referencia_carrera(
    id="ING-CIVIL",
    nombre="Ingeniería Civil",
    facultad_id="FCMF",
    bloque_id="B1",
    alias=["civil", "ing civil", "ingenieria civil"]
)

REFERENCIAS_B1["ING-INDUSTRIAL"] = crear_referencia_carrera(
    id="ING-INDUSTRIAL",
    nombre="Ingeniería Industrial",
    facultad_id="FII",
    bloque_id="B1",
    alias=["industrial", "ing industrial", "ingenieria industrial"]
)

REFERENCIAS_B1["ING-MECA"] = crear_referencia_carrera(
    id="ING-MECA",
    nombre="Ingeniería Mecatrónica",
    facultad_id="FII",
    bloque_id="B1",
    alias=["mecatronica", "ing mecatronica", "ingenieria mecatronica"]
)

REFERENCIAS_B1["ING-S_INFO"] = crear_referencia_carrera(
    id="ING-S_INFO",
    nombre="Sistemas de Información",
    facultad_id="FII",
    bloque_id="B1",
    alias=["sistemas", "sistemas de informacion", "ing. sistemas"]
)

REFERENCIAS_B1["SOFT"] = crear_referencia_carrera(
    id="SOFT",
    nombre="Ingeniería en Software",
    facultad_id="FCMF",
    bloque_id="B1",
    alias=["software", "ing software", "ingenieria en software"]
)

REFERENCIAS_B1["TIC"] = crear_referencia_carrera(
    id="TIC",
    nombre="Tecnologías de la Información",
    facultad_id="FCMF",
    bloque_id="B1",
    alias=["tecnologias de la informacion", "tics", "tic"]
)

REFERENCIAS_B1["TELEMA"] = crear_referencia_carrera(
    id="TELEMA",
    nombre="Telemática",
    facultad_id="FII",
    bloque_id="B1",
    alias=["telematica", "ingeniería en telemática", "ing. telematica"]
)

## Bloque 2 - Ciencias de la Bioindustria y Recursos Naturales

In [8]:
REFERENCIAS_B2["AGRONO"] = crear_referencia_carrera(
    id="AGRONO",
    nombre="Agronomía",
    facultad_id="FCCAGRA",
    bloque_id="B2",
    alias=["agronomía", "agro"]
)

REFERENCIAS_B2["AGROPE"] = crear_referencia_carrera(
    id="AGROPE",
    nombre="Agropecuaria",
    facultad_id="FCCAGRA",
    bloque_id="B2",
    alias=["agropecuaria"]
)

REFERENCIAS_B2["ALIMENTOS"] = crear_referencia_carrera(
    id="ALIMENTOS",
    nombre="Alimentos",
    facultad_id="FIQ",
    bloque_id="B2",
    alias=["alimento", "ingeniería en alimentos", "alimentos", "ing en alimentos"]
)

REFERENCIAS_B2["BIO"] = crear_referencia_carrera(
    id="BIO",
    nombre="Biología",
    facultad_id="FCCNN",
    bloque_id="B2",
    alias=["biología", "ingeniería en biología", "bio", "ing en biología"]
)

REFERENCIAS_B2["GEO"] = crear_referencia_carrera(
    id="GEO",
    nombre="Geología",
    facultad_id="FCCQQ",
    bloque_id="B2",
    alias=["geo", "geología"]
)

REFERENCIAS_B2["AMBIENTAL"] = crear_referencia_carrera(
    id="AMBIENTAL",
    nombre="Ingeniería Ambiental",
    facultad_id="FCCNN",
    bloque_id="B2",
    alias=["ambiental", "ingenieria ambiente", "ingeniería ambiental", "ing ambiente", "ing ambiental"]
)

REFERENCIAS_B2["ING-PRODUCCION"] = crear_referencia_carrera(
    id="ING-PRODUCCION",
    nombre="Ingeniería de la Producción",
    facultad_id="FIQ",
    bloque_id="B2",
    alias=["ingeniería de producción", "ing producción","ingeniería de la producción"]
)

REFERENCIAS_B2["QUIMICA"] = crear_referencia_carrera(
    id="QUIMICA",
    nombre="Ingeniería Química",
    facultad_id="FIQ",
    bloque_id="B2",
    alias=["ingeniería química", "quimica", "ing quimica"]
)

REFERENCIAS_B2["VETERINARIA"] = crear_referencia_carrera(
    id="VETERINARIA",
    nombre="Medicina Veterinaria",
    facultad_id="FMVZ",
    bloque_id="B2",
    alias=["veterinaria", "medi veterinaria", "medicina veterinaria"]
)

##BLOQUE 3 - Diseño y Creatividad


In [9]:

REFERENCIAS_B3["DI"] = crear_referencia_carrera(
    id="DI",
    nombre="Diseño de interiores",
    facultad_id="FAU",
    bloque_id="B3",
    alias=[" Interiores, Diseño de interiores, Arquitectura de interiores"]
)

REFERENCIAS_B3["DG"] = crear_referencia_carrera(
    id="DG",
    nombre="Diseño Gráfico",
    facultad_id="FACSO",
    bloque_id="B3",
    alias=["Gráfico", "Artes", "Diseño Grafico", "Artes Gráficas"]
)


## BLOQUE 4 - Ciencias Políticas y Sociales

In [11]:
REFERENCIAS_B4["C-Políticas"] = crear_referencia_carrera(
    id="C-Políticas",
    nombre="Ciencias Políticas",
    facultad_id="FCJSP",
    bloque_id="B4",
    alias=["Políticas", "Ciencia Política", "Política", "Gobierno", "Ciencias Políticas"]
)

REFERENCIAS_B4["Comunicación"] = crear_referencia_carrera(
    id="Comunicación",
    nombre="Comunicación",
    facultad_id="FACSO",
    bloque_id="B4",
    alias=["Comunicación Social", "Periodismo", "Medios", "Comunicador", "Comunicacion"]
)

REFERENCIAS_B4["Derecho"] = crear_referencia_carrera(
    id="Derecho",
    nombre="Derecho",
    facultad_id="FJCSP",
    bloque_id="B4",
    alias=["Abogacía", "Leyes", "Jurisprudencia", "Ciencias Jurídicas", "Abogado"]
)

REFERENCIAS_B4["Educación-Básica"] = crear_referencia_carrera(
    id="Educación-Básica",
    nombre="Educación Básica",
    facultad_id="FILO",
    bloque_id="B4",
    alias=["Básica", "Educación Básica", "Docencia", "Pedagogía", "Educacion Basica"]
)

REFERENCIAS_B4["Educación-Inicial"] = crear_referencia_carrera(
    id="Educación-Inicial",
    nombre="Educación Inicial",
    facultad_id="FILO",
    bloque_id="B4",
    alias=["Inicial", "Párvulos", "Preescolar", "Estimulación Temprana", "Educacion Inicial"]
)

REFERENCIAS_B4["Deporte"] = crear_referencia_carrera(
    id="Deporte",
    nombre="Entrenamiento Deportivo",
    facultad_id="FACAF",
    bloque_id="B4",
    alias=["Deportes", "Entrenador", "Actividad Física", "Cultura Física", "Entrenamiento Deportivo"]
)

REFERENCIAS_B4["Gastronomía"] = crear_referencia_carrera(
    id="Gastronomía",
    nombre="Gastronomía",
    facultad_id="FIQ",
    bloque_id="B4",
    alias=["Cocina", "Chef", "Arte Culinario", "Alimentos", "Gastronomia"]
)

REFERENCIAS_B4["Pedagogía-Física-Deporte"] = crear_referencia_carrera(
    id="Pedagogía-Física-Deporte",
    nombre="Pedagogía de la Actividad Física y Deporte",
    facultad_id="FACAF",
    bloque_id="B4",
    alias=["Educación Física", "Pedagogía Deportiva", "Docencia Deportiva", "Actividad Física", "Pedagogia Actividad Fisica"]
)

REFERENCIAS_B4["Pedagogía-Filosofía"] = crear_referencia_carrera(
    id="Pedagogía-Filosofía",
    nombre="Pedagogía de la Filosofía",
    facultad_id="FILO",
    bloque_id="B4",
    alias=["Filosofía", "Pedagogía Filosófica", "Docencia Filosofía", "Artes y Humanidades", "Pedagogia Filosofia"]
)

REFERENCIAS_B4["Pedagogía-Historia-Ciencias"] = crear_referencia_carrera(
    id="Pedagogía-Historia-Ciencias",
    nombre="Pedagogía de la Historia y Ciencias Sociales",
    facultad_id="FILO",
    bloque_id="B4",
    alias=["Historia", "Ciencias Sociales", "Docencia Historia", "Estudios Sociales", "Pedagogia Historia"]
)

REFERENCIAS_B4["Pedagogía-Informática"] = crear_referencia_carrera(
    id="Pedagogía-Informática",
    nombre="Pedagogía de la Informática",
    facultad_id="FILO",
    bloque_id="B4",
    alias=["Informática", "Computación", "TIC", "Docencia Informática", "Tecnología Educativa", "Pedagogia Informatica"]
)

REFERENCIAS_B4["Pedagogía-Lengua-Literaria"] = crear_referencia_carrera(
    id="Pedagogía-Lengua-Literaria",
    nombre="Pedagogía de la Lengua y Literatura",
    facultad_id="FILO",
    bloque_id="B4",
    alias=["Lengua", "Literatura", "Docencia Lenguaje", "Letras", "Pedagogia Lengua"]
)

REFERENCIAS_B4["Pedagogía-Matemáticas-Física"] = crear_referencia_carrera(
    id="Pedagogía-Matemáticas-Física",
    nombre="Pedagogía de Matemáticas y Física",
    facultad_id="FILO",
    bloque_id="B4",
    alias=["Matemáticas", "Física", "Docencia Matemáticas", "Ciencias Experimentales", "Pedagogia Matematicas"]
)

REFERENCIAS_B4["Pedagogía-Química-Biología"] = crear_referencia_carrera(
    id="Pedagogía-Química-Biología",
    nombre="Pedagogía de Química y Biología",
    facultad_id="FILO",
    bloque_id="B4",
    alias=["Química", "Biología", "Docencia Química", "Laboratorio", "Ciencias Naturales", "Pedagogia Quimica"]
)

REFERENCIAS_B4["Pedagogía-Artes-Humanidades"] = crear_referencia_carrera(
    id="Pedagogía-Artes-Humanidades",
    nombre="Pedagogía de las Artes",
    facultad_id="FILO",
    bloque_id="B4",
    alias=["Artes", "Humanidades", "Escénicas", "Teatro", "Danza", "Expresión Artística", "Pedagogia Artes"]
)

REFERENCIAS_B4["Pedagogía-Idiomas"] = crear_referencia_carrera(
    id="Pedagogía-Idiomas",
    nombre="Pedagogía del Idioma Inglés",
    facultad_id="FILO",
    bloque_id="B4",
    alias=["Inglés", "Idiomas", "Docencia Inglés", "Lenguas Extranjeras", "Bilingüe", "Pedagogia Idiomas"]
)

REFERENCIAS_B4["Psicología-Educativa"] = crear_referencia_carrera(
    id="Psicología-Educativa",
    nombre="Psicología Educativa",
    facultad_id="FCP",
    bloque_id="B4",
    alias=["Psicología", "Psicopedagogía", "Orientación", "Psicólogo Educativo", "Psicologia Educativa"]
)

REFERENCIAS_B4["Publicidad"] = crear_referencia_carrera(
    id="Publicidad",
    nombre="Publicidad",
    facultad_id="FACSO",
    bloque_id="B4",
    alias=["Marketing", "Propaganda", "Mercadeo", "Branding", "Publicidad y Marketing"]
)

REFERENCIAS_B4["Sociología"] = crear_referencia_carrera(
    id="Sociología",
    nombre="Sociología",
    facultad_id="FJCSP",
    bloque_id="B4",
    alias=["Ciencias Sociales", "Investigación Social", "Desarrollo Social", "Análisis Social", "Sociologia"]
)

##BLOQUE 5: Negocios y Economía Global

In [12]:
REFERENCIAS_B5["A-Empresas"] = crear_referencia_carrera(
    id="A-Empresas",
    nombre="Administración de Empresas",
    facultad_id="FCCAA",
    bloque_id="B5",
    alias=["Empresas", "Administración", "Negocios", "Gestión Empresarial", "Administracion"]
)

REFERENCIAS_B5["Comercio-E"] = crear_referencia_carrera(
    id="Comercio-E",
    nombre="Comercio Exterior",
    facultad_id="FCCAA",
    bloque_id="B5",
    alias=["Comercio", "Aduanas", "Importación", "Exportación", "Logística Internacional"]
)

REFERENCIAS_B5["Contabilidad-Auditoría"] = crear_referencia_carrera(
    id="Contabilidad-Auditoría",
    nombre="Contabilidad y Auditoría",
    facultad_id="FCCAA",
    bloque_id="B5",
    alias=["Contabilidad", "Auditoría", "Contador", "Financiero", "Tributación"]
)

REFERENCIAS_B5["Economía"] = crear_referencia_carrera(
    id="Economía",
    nombre="Economía",
    facultad_id="FCE",
    bloque_id="B5",
    alias=["Economista", "Análisis Económico", "Desarrollo", "Macroeconomía", "Microeconomía"]
)

REFERENCIAS_B5["Economía-Inter"] = crear_referencia_carrera(
    id="Economía-Inter",
    nombre="Economía Internacional",
    facultad_id="FCE",
    bloque_id="B5",
    alias=["Economía Internacional", "Comercio Global", "Finanzas Internacionales", "Globalización", "Economia Internacional"]
)

REFERENCIAS_B5["Finanzas"] = crear_referencia_carrera(
    id="Finanzas",
    nombre="Finanzas",
    facultad_id="FCCAA",
    bloque_id="B5",
    alias=["Financiero", "Inversiones", "Banca", "Mercados Financieros", "Gestión Financiera"]
)

REFERENCIAS_B5["Gestión-Gerencial"] = crear_referencia_carrera(
    id="Gestión-Gerencial",
    nombre="Gestión de la Información Gerencial",
    facultad_id="FCCAA",
    bloque_id="B5",
    alias=["Gestión", "Gerencial", "Informática Gerencial", "TIC", "Administración Tecnológica"]
)

REFERENCIAS_B5["Mercadotecnia"] = crear_referencia_carrera(
    id="Mercadotecnia",
    nombre="Mercadotecnia",
    facultad_id="FCCAA",
    bloque_id="B5",
    alias=["Marketing", "Mercadeo", "Ventas", "Publicidad", "Comercialización"]
)

REFERENCIAS_B5["Negocios-Internacionales"] = crear_referencia_carrera(
    id="Negocios-Internacionales",
    nombre="Negocios Internacionales",
    facultad_id="FCCAA",
    bloque_id="B5",
    alias=["Negocios", "Internacionales", "Comercio", "Global", "Relaciones Internacionales", "Negocios Internacionales"]
)

REFERENCIAS_B5["Turismo"] = crear_referencia_carrera(
    id="Turismo",
    nombre="Turismo",
    facultad_id="FCCAA",
    bloque_id="B5",
    alias=["Hotelería", "Agencias de Viajes", "Sostenibilidad Turística", "Guiado Turístico"]
)

##BLOQUE 6 - Ciencias de la Salud Humana

In [13]:
REFERENCIAS_B6["Bio-Farm"] = crear_referencia_carrera(
    id="Bio-Farm",
    nombre="Bioquímica y Farmacia",
    facultad_id="FCCQQ",
    bloque_id="B6",
    alias=["Bioquímica", "Farmacia", "Biofarmacia", "Química Farmacéutica", "Laboratorio Clínico", "Biotecnología"]
)

REFERENCIAS_B6["Enfermería"] = crear_referencia_carrera(
    id="Enfermería",
    nombre="Enfermería",
    facultad_id="FCCMM",
    bloque_id="B6",
    alias=["Enfermero", "Licenciatura en Enfermería", "Cuidados de Salud", "Atención Primaria", "Salud Comunitaria"]
)

REFERENCIAS_B6["Fonoaudiología"] = crear_referencia_carrera(
    id="Fonoaudiología",
    nombre="Fonoaudiología",
    facultad_id="FCCMM",
    bloque_id="B6",
    alias=["Logopedia", "Terapia del Lenguaje", "Audición", "Fonoaudiólogo"]
)

REFERENCIAS_B6["Medicina"] = crear_referencia_carrera(
    id="Medicina",
    nombre="Medicina",
    facultad_id="FCCMM",
    bloque_id="B6",
    alias=["Médico", "Medicina General", "Ciencias Médicas", "Salud Humana", "Atención Médica"]
)

REFERENCIAS_B6["Nutrición-Die"] = crear_referencia_carrera(
    id="Nutrición-Die",
    nombre="Nutrición y Dietética",
    facultad_id="FCCMM",
    bloque_id="B6",
    alias=["Nutrición", "Dietética", "Nutricionista", "Alimentación Saludable", "Dietoterapia"]
)

REFERENCIAS_B6["Obstetricia"] = crear_referencia_carrera(
    id="Obstetricia",
    nombre="Obstetricia",
    facultad_id="FCCMM",
    bloque_id="B6",
    alias=["Obstetra", "Matrona", "Salud Reproductiva", "Maternidad", "Ginecología"]
)

REFERENCIAS_B6["Odontología"] = crear_referencia_carrera(
    id="Odontología",
    nombre="Odontología",
    facultad_id="FO",
    bloque_id="B6",
    alias=["Odontólogo", "Dentista", "Cirujano Dentista", "Salud Bucal",]
)

REFERENCIAS_B6["Psicología"] = crear_referencia_carrera(
    id="Psicología",
    nombre="Psicología",
    facultad_id="FCP",
    bloque_id="B6",
    alias=["Psicólogo", "Psicología", "Comportamiento Humano", "Psicoterapia", "Salud Mental"]
)

REFERENCIAS_B6["T-Ocupacional"] = crear_referencia_carrera(
    id="T-Ocupacional",
    nombre="Terapia Ocupacional",
    facultad_id="FCCMM",
    bloque_id="B6",
    alias=["Terapia Ocupacional", "Terapeuta Ocupacional", "Rehabilitación Física", "Capacitación Funcional", "Ocupacional"]
)

REFERENCIAS_B6["T-Respiratoria"] = crear_referencia_carrera(
    id="T-Respiratoria",
    nombre="Terapia Respiratoria",
    facultad_id="FCCMM",
    bloque_id="B6",
    alias=["Terapia Respiratoria", "Terapeuta Respiratorio", "Cardio-Respiratorio", "Neumología", "Cuidados Respiratorios", "Respiratoria"]
)

#6) Procesos Académicos

In [14]:
nivelacion = crear_entidad_simple(
    "NIV", "Curso de Nivelación",
     ["nivelación", "curso de nivelación", "nivelacion"])

inscripcion = crear_entidad_simple(
    "INS", "Inscripción y Postulación",
     ["inscripción", "inscribirme", "registrarme", "postulación", "postular"])

matricula = crear_entidad_simple(
    "MAT", "Matrícula",
     ["matrícula", "matricula", "matricularme"])

admision = crear_entidad_simple(
    "ADM", "Admisión",
    ["admisión", "admision", "ingreso", "proceso de admisión"])

cupo_aceptado = crear_entidad_simple(
    "CUP", "Aceptación de Cupo",
    ["cupo", "cupo aceptado", "aceptar cupo", "aceptación de cupo"])

registro_nacional = crear_entidad_simple(
    "REG_NAC", "Registro Nacional (MINEDUC)",
     ["registro nacional", "registro minedec", "registro mineduc"])

evaluacion = crear_entidad_simple(
    "EVAL", "Evaluación de Admisión",
    ["evaluación", "examen de admisión", "examen", "prueba de admisión"])

procesos = [
    nivelacion,
    inscripcion,
    matricula,
    admision,
    cupo_aceptado,
    registro_nacional,
    evaluacion
    ]

#7) Documentos

In [15]:
cedula = crear_entidad_simple(
    "CED", "Cédula de identidad",
     ["cédula", "cedula", "documento de identidad"])

titulo = crear_entidad_simple(
    "TIT", "Título de Bachiller",
     ["título", "bachiller", "título de bachiller"])

pasaporte = crear_entidad_simple(
    "PAS", "Pasaporte",
     ["pasaporte"])

documentos = [
    cedula,
    titulo,
    pasaporte
    ]

#8) Patrones (Expresiones Regulares)

In [16]:
patrones = {
    "patrones": {
        #Cédula 10 dígitos
        "cedula": r"\b\d{10}\b",
        #Fechas en formato numérico (DD/MM/AAAA) o textual (DD de MES)
        "fecha": r"\b(\d{1,2}[-/]\d{1,2}[-/]\d{2,4}|\d{1,2}\s+de\s+[a-z]+)\b",
        #Palabra relacionada con "bloque" seguida de un número del 1 al 6 (en dígitos o letras)
        "bloque": r"\bbloque\s*(1|2|3|4|5|6|uno|dos|tres|cuatro|cinco|seis)\b",
    }
}

#9) Agrupar

## Función para limpiar alias

In [18]:
def limpiar_alias(texto):
    # Convertimos a minúsculas y quitar puntos
    texto_limpio = texto.lower().replace(".", "")

    # Remover tildes y caracteres especiales usando unicodedata
    texto_limpio = unicodedata.normalize('NFKD', texto_limpio)
    texto_limpio = "".join([c for c in texto_limpio if not unicodedata.combining(c)])

    return texto_limpio.strip()

## Agrupar las carreras registrada en cada bloque en un solo diccionario

In [19]:
CARRERAS = {
    **REFERENCIAS_B1,
    **REFERENCIAS_B2,
    **REFERENCIAS_B3,
    **REFERENCIAS_B4,
    **REFERENCIAS_B5,
    **REFERENCIAS_B6,
}

## Bucle para limpiar_alias del diccionario CARRERAS

In [20]:
for id_carrera, datos_carrera in CARRERAS.items():
    datos_carrera["alias"] = [limpiar_alias(a) for a in datos_carrera["alias"]]

## Unir todo en un solo diccionario

In [21]:
entidades = {
    "facultades": FACULTADES,
    "carreras": CARRERAS,
    "procesos": procesos,
    "documentos": documentos,
    "patrones": patrones,
}

#10) Validar

In [22]:
print(f"Facultades: {len(entidades['facultades'])} (de 17 totales en la UG)")
print(f"Carreras: {len(entidades['carreras'])} (de 59 totales en la UG)")
print(f"Procesos: {len(entidades['procesos'])}")
print(f"Documentos: {len(entidades['documentos'])}")
print(f"Patrones RegEx: {len(entidades['patrones']['patrones'])}")

Facultades: 17 (de 17 totales en la UG)
Carreras: 59 (de 59 totales en la UG)
Procesos: 7
Documentos: 3
Patrones RegEx: 3


#11) Descargar archivo

In [23]:
with open(ARCHIVO_SALIDA, "w", encoding="utf-8") as archivo:
    json.dump(entidades, archivo, indent=4, ensure_ascii=False)

# Descarga de archivo
files.download(ARCHIVO_SALIDA)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>